In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc

import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
import warnings
import pandas as pd
import statsmodels.formula.api as smf

In [ ]:
from scipy import stats

In [ ]:
import warnings
warnings.filterwarnings("ignore")

In [ ]:
sc.settings.verbosity = 4
sc.logging.print_header()
sc.settings.set_figure_params(dpi=300, facecolor='white', format = 'pdf', vector_friendly = True)

In [ ]:
umap_cmap = sns.blend_palette(['lightgrey', 'xkcd:sapphire'], as_cmap = True)

In [ ]:
figure = "Figure_7"

In [ ]:
sc.settings.figdir = './Figure_plots/'+figure

# Load and prepare dataset

In [ ]:
adata = sc.read_h5ad('./h5ad/analysis_250528_f/Smed_L78-L47_20250523_Annotated.h5ad')
adata

In [ ]:
sc.pl.umap(adata, color = 'score_DGE_G1', cmap = umap_cmap, size = 10)

In [ ]:
sc.pl.umap(adata, color = 'score_DGE_G2', cmap = 'Purples', size = 10)

In [ ]:
# Violin plot for the broad groups with conditions side by side
# Subset to the conditions of interest
subset = adata[adata.obs['Condition'].isin(['GFP','Cdh1','H2B'])].copy()
subset.obs['Condition_broad'] = (
    subset.obs['broad_names2'].astype(str) + '_' + subset.obs['Condition'].astype(str)
)

# Define the color palette
cond_colors = {
    'GFP' : 'slategray',  
    'Cdh1': 'dodgerblue',   
    'H2B' : 'darkviolet'   
}
subset.obs['Condition_broad'] = subset.obs['Condition_broad'].astype('category')
cat_order = subset.obs['Condition_broad'].cat.categories
palette = [cond_colors[cat.split('_')[-1]] for cat in cat_order]

# Plot violin
with plt.rc_context({'figure.figsize': (15, 5)}):
    sc.pl.violin(
        subset,
        'score_DGE_G2',
        groupby='Condition_broad',
        jitter=False,
        rotation=90,
        stripplot=False,
        palette=palette
    )


# linear mixed model

## functions

In [ ]:
def run_lmm_summary(
    adata,
    celltype_col='broad_names2',
    score_col='score_DGE_G2',
    condition_col='Condition',
    sample_col='Sample',
    reference='GFP',
    celltypes=None
):

    warnings.filterwarnings("ignore")
    results_list = []

    # Determine which cell types to loop over
    if celltypes is None:
        celltypes_to_use = adata.obs[celltype_col].unique()
    else:
        celltypes_to_use = [ct for ct in celltypes if ct in adata.obs[celltype_col].unique()]

    for ct in celltypes_to_use:
        try:
            df = adata.obs[(adata.obs['Experiment'] == 'RNAi') &
                           (adata.obs[celltype_col] == ct)][[score_col, condition_col, sample_col]].copy()
            
            df[condition_col] = df[condition_col].astype("category")
            df[condition_col] = df[condition_col].cat.set_categories(sorted(df[condition_col].unique()))
            
            if reference in df[condition_col].cat.categories:
                df[condition_col] = df[condition_col].cat.reorder_categories(
                    [reference] + [c for c in df[condition_col].cat.categories if c != reference],
                    ordered=True
                )
            
            df[sample_col] = df[sample_col].astype("category").cat.remove_unused_categories()
            
            # Fit LMM
            model = smf.mixedlm(f"{score_col} ~ {condition_col}", df, groups=df[sample_col])
            result = model.fit()
            
            # Extract condition p-values
            row = {'CellType': ct}
            for cond, pval in result.pvalues.items():
                if cond != 'Intercept':
                    clean_name = cond.replace(f"{condition_col}[T.","").replace("]","")
                    row[clean_name] = pval
            results_list.append(row)
        
        except Exception as e:
            print(f"Skipped {ct} due to error: {e}")
            results_list.append({'CellType': ct})
    
    # Build dataframe
    summary_df = pd.DataFrame(results_list)
    for col in summary_df.columns:
        if col != 'CellType':
            summary_df[col] = summary_df[col].apply(lambda x: f"{x:.6f}" if pd.notnull(x) else "")
    
    return summary_df

In [ ]:
# subset for boxplot to visualise the score per sample
def prepare_broad_subset_v2(
    adata, 
    celltype_col='broad_names2',   
    sample_col='Sample',           
    li=None
):
    # Subset the relevant samples
    subset = adata[
        adata.obs[sample_col].isin(['GFP_1', 'GFP_2', 'Cdh1_1', 'Cdh1_2', 'H2B_1', 'H2B_2'])
    ].copy()

    # Subset specific cell types 
    if li is not None:
        subset = subset[subset.obs[celltype_col].isin(li)].copy()
        clusters = li
    else:
        clusters = sorted(subset.obs[celltype_col].unique())

    suffix_order = ['GFP_1', 'GFP_2', 'Cdh1_1', 'Cdh1_2', 'H2B_1', 'H2B_2']

    # Create categorical order
    cat_order = [
        f"{cluster}_{suffix}"
        for cluster in clusters
        for suffix in suffix_order
    ]

    subset.obs['Condition_broad'] = (
        subset.obs[celltype_col].astype(str) + '_' +
        subset.obs[sample_col].astype(str)
    )

    subset.obs['Condition_broad'] = pd.Categorical(
        subset.obs['Condition_broad'],
        categories=cat_order,
        ordered=True
    )

    return subset, cat_order, suffix_order

In [ ]:
# boxplot
def plot_violin_box(subset, cat_order, suffix_order, file_name, score ='score_DGE_G2', lenght=11, height=6 ):

    cond_colors = {'GFP_1' : 'slategray', 'GFP_2' : 'darkgray', 
                   'Cdh1_1': 'dodgerblue', 'Cdh1_2': 'skyblue', 
                   'H2B_1' : 'darkviolet', 'H2B_2' : 'mediumorchid'
    }

    # input data
    data = []
    colors = []

    for cat in cat_order:
        values = subset[subset.obs['Condition_broad'] == cat].obs[score]
        data.append(values)
        suffix = '_'.join(cat.split('_')[-2:])
        colors.append(cond_colors[suffix])

    # X positions
    group_size = len(suffix_order)
    gap = 1

    positions = []
    for i in range(len(data)):
        cluster_index = i // group_size
        pos = i + cluster_index * gap
        positions.append(pos)

    # Plot 
    fig, ax = plt.subplots(figsize=(lenght, height))

    box = ax.boxplot(
        data,
        positions=positions,
        widths=0.6,
        patch_artist=True,
        showfliers=False, 
        medianprops=dict(color='black', linewidth=2)
    )

    for patch, color in zip(box['boxes'], colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.8)

    # Jitter
    for i, values in enumerate(data):
        x = np.random.normal(positions[i], 0.02, size=len(values))
        ax.scatter(
            x,
            values,
            color=colors[i],  
            alpha=0.4,
            s=1,
            zorder=2, 
            rasterized=True
        )

    ax.set_xticks(positions)
    ax.set_xticklabels(cat_order, rotation=90)

    ax.set_ylabel(score)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    ax.grid(False)
    ax.yaxis.grid(True)

    plt.tight_layout()
    plt.savefig(f'./Figure_plots/{figure}/{file_name}', dpi=300 )
    plt.show()

    del subset # delete this variable to free RAM

## LMM for categories split according to the neoblast scores

In [ ]:
# create a new column to split the broad groups into progenitors/ differentiated cells according to the neoblast_score
# thresholds: 
#score > 0.259: 'Neoblast'
# score < 0.171:'Differentiated'
# score between: 'Progenitor'

# for broad_group = 'Neoblasts' => score > 0.259 = neoblasts, score < 0.259 = progenitors
# for all other broad_group => score > 0.171 = progenitor_ + cell type name, score < 0.171 = diff_ +  cell type name

def assign_cell_state(row, score_col='neoblast_score', group_col='broad_names'):
    score = row[score_col]
    group = row[group_col]
    
    if group == 'neoblasts':
        if score > 0.259:
            return 'neoblasts'
        else:
            return 'progenitor_neoblasts'
    else:
        if score > 0.171:
            return f'progenitor_{group}'
        else:
            return f'diff_{group}'
    
adata.obs['cell_state'] = adata.obs.apply(assign_cell_state, axis=1)

In [ ]:
# G2 score

In [ ]:
summary_df = run_lmm_summary(
    adata,
    celltype_col='cell_state',
    score_col='score_DGE_G2',
    condition_col='Condition',
    sample_col='Sample',
    reference='GFP',
)

summary_df

In [ ]:
summary_df['Cdh1'] = summary_df['Cdh1'].astype(float)
summary_df['H2B'] = summary_df['H2B'].astype(float)
li_ct_sig = list(summary_df[(summary_df['Cdh1'] < 0.025) | (summary_df['H2B'] < 0.025)]['CellType'])

In [ ]:
li_ct_sig

In [ ]:
# significant cell types
# li is used to set the order
li = [ 'progenitor_neoblasts',  'progenitor_neurons',  'progenitor_muscle', 'progenitor_parenchymal', 
      'progenitor_epidermis',  'progenitor_phagocytes', 'progenitor_secretory', 'progenitor_protonephridia', 
      'diff_protonephridia',   'progenitor_pharynx',  'progenitor_unannotated']
subset, cat_order, suffix_order = prepare_broad_subset_v2(adata, celltype_col='cell_state', li=li)


plot_violin_box(
    subset,
    cat_order,
    suffix_order,
    file_name="broad_groups_score_G2_sig.svg", 
    lenght=20, height=8
)

In [ ]:
li_ct2 = [ i for i in adata.obs['cell_state'].astype('category').cat.categories if i not in li_ct_sig]
li_ct2

In [ ]:
# cell types without significant difference
li= [ 'neoblasts', 'diff_neurons', 'diff_muscle', 'diff_parenchymal','diff_epidermis', 
      'diff_phagocytes',  'diff_secretory',  'diff_pharynx',  'progenitor_psd+ cells', 'diff_psd+ cells',
 'progenitor_basal goblet', 'diff_basal goblet', 'diff_unannotated']

subset, cat_order, suffix_order = prepare_broad_subset_v2(adata, celltype_col='cell_state', li=li)

plot_violin_box(
    subset,
    cat_order,
    suffix_order,
    file_name="broad_groups_score_G2_non_sig"
)

In [ ]:
# G1 score

In [ ]:
summary_df = run_lmm_summary(
    adata,
    celltype_col='cell_state',
    score_col='score_DGE_G1',
    condition_col='Condition',
    sample_col='Sample',
    reference='GFP',
#    celltypes=celltype_list
)

summary_df

In [ ]:
summary_df[['Cdh1', 'H2B']] = summary_df[['Cdh1', 'H2B']].replace('', np.nan)
summary_df['Cdh1'] = summary_df['Cdh1'].astype(float)
summary_df['H2B'] = summary_df['H2B'].astype(float)
li_ct_sig = list(summary_df[(summary_df['Cdh1'] < 0.025) | (summary_df['H2B'] < 0.025)]['CellType'])

In [ ]:
li_ct_sig

In [ ]:
# significant cell types
li = [ 'progenitor_muscle', 'progenitor_phagocytes',
 'progenitor_secretory',
 'diff_unannotated']
subset, cat_order, suffix_order = prepare_broad_subset_v2(adata, celltype_col='cell_state', li=li)

plot_violin_box(
    subset,
    cat_order,
    suffix_order,
     score ='score_DGE_G1',
    file_name="broad_groups_score_G1_sig"
)

In [ ]:
li_ct2 = [ i for i in adata.obs['cell_state'].astype('category').cat.categories if i not in li_ct_sig]
li_ct2

In [ ]:
# cell types without significant difference
li= [  'neoblasts',  'progenitor_neoblasts', 'progenitor_neurons', 'diff_neurons', 'diff_muscle',  
      'progenitor_parenchymal', 'diff_parenchymal', 'progenitor_epidermis', 'diff_epidermis',  
     'diff_phagocytes',  'diff_secretory', 'diff_protonephridia',  'progenitor_pharynx', 'diff_pharynx',  
     'progenitor_psd+ cells',  'diff_psd+ cells', 
     'progenitor_basal goblet', 'diff_basal goblet', 'progenitor_unannotated']

subset, cat_order, suffix_order = prepare_broad_subset_v2(adata, celltype_col='cell_state', li=li)

plot_violin_box(
    subset,
    cat_order,
    suffix_order,
         score ='score_DGE_G1',
    file_name="broad_groups_score_G1_non_sig"
)